In [5]:
# For tips on running notebooks in Google Colab, see
# https://pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\| **Quickstart** \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Quickstart
==========

This section runs through the API for common tasks in machine learning.
Refer to the links in each section to dive deeper.

Working with data
-----------------

PyTorch has two [primitives to work with
data](https://pytorch.org/docs/stable/data.html):
`torch.utils.data.DataLoader` and `torch.utils.data.Dataset`. `Dataset`
stores the samples and their corresponding labels, and `DataLoader`
wraps an iterable around the `Dataset`.


In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader#Dataloader为Datasets打包一个迭代器
from torchvision import datasets
from torchvision.transforms import ToTensor

PyTorch offers domain-specific libraries such as
[TorchText](https://pytorch.org/text/stable/index.html),
[TorchVision](https://pytorch.org/vision/stable/index.html), and
[TorchAudio](https://pytorch.org/audio/stable/index.html), all of which
include datasets. For this tutorial, we will be using a TorchVision
dataset.

The `torchvision.datasets` module contains `Dataset` objects for many
real-world vision data like CIFAR, COCO ([full list
here](https://pytorch.org/vision/stable/datasets.html)). In this
tutorial, we use the FashionMNIST dataset. Every TorchVision `Dataset`
includes two arguments: `transform` and `target_transform` to modify the
samples and labels respectively.




//自增笔记📒：
- torchtext：做NLP
- torchvision：做CV
- torchaudio：做语音处理

FashionMNIST是一个服装分类数据集
- 所有TorchVision都有两个关键参数：
- 1. transform：用来处理/转换图片样本，归一化，resize等
- 2. target_transform：用来处理标签，比如把标签转成张量

In [7]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(#每个torchvision的datasets均包含两个参数：transform和target_transform，分别用于对样本数据和标签进行处理
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),#这一步是把图片变成tensor，从（高，宽）-》（通道，高，宽）+归一化（除以像素总数255）
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

We pass the `Dataset` as an argument to `DataLoader`. This wraps an
iterable over our dataset, and supports automatic batching, sampling,
shuffling and multiprocess data loading. Here we define a batch size of
64, i.e. each element in the dataloader iterable will return a batch of
64 features and labels.



//自增笔记📒：
DataLoader可以自动分组（批处理）、【按规则取样、随机打乱数据】（此两者经常结合）、多进程加速加载数据

In [8]:
batch_size = 64#一批处理64条数据

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")#input:【批数量，通道，high，width】
    print(f"Shape of y: {y.shape} {y.dtype}")#output:标签
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


Read more about [loading data in PyTorch](data_tutorial.html).


------------------------------------------------------------------------


Creating Models
===============

To define a neural network in PyTorch, we create a class that inherits
from
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
We define the layers of the network in the `__init__` function and
specify how data will pass through the network in the `forward`
function. To accelerate operations in the neural network, we move it to
the
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [10]:
device = torch.accelerator.current_accelerator().type if torch.cuda.is_available() else "cpu"#若加速器可用就使用加速器，若不可用就用CPU
print(f"Using {device} device")#输出使用的设备

# Define model
class NeuralNetwork(nn.Module):#继承基类模型类
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()#把多维图像flatten为一维向量28*28-784
        self.linear_relu_stack = nn.Sequential(#按顺序堆叠网络层
            nn.Linear(28*28, 512),#第一层全连接
            nn.ReLU(),#激活函数（非线性激活）
            nn.Linear(512, 512),#第二层全连接
            nn.ReLU(),
            nn.Linear(512, 10)#输出层，输出10维，用于10分类
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)#把模型搬到CPU/GPU（各个厂商不同，cuda/mps/mtia均属GPU）上运行
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Read more about [building neural networks in
PyTorch](buildmodel_tutorial.html).


------------------------------------------------------------------------


Optimizing the Model Parameters
===============================

To train a model, we need a [loss
function](https://pytorch.org/docs/stable/nn.html#loss-functions) and an
[optimizer](https://pytorch.org/docs/stable/optim.html).


In [11]:
loss_fn = nn.CrossEntropyLoss()#标准CE损失函数
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)#定义优化器，随机梯度下降（传入模型的所有可训练参数，learning rate）

In a single training loop, the model makes predictions on the training
dataset (fed to it in batches), and backpropagates the prediction error
to adjust the model\'s parameters.


//自增笔记📒：
单次训练循环 一套固定动作：
1. 拿一个 batch 数据进来
2. 前向传播：做预测（forward）
3. 算损失（loss）
4. 反向传播：算梯度（backward）
5. 优化器更新参数（optimizer.step ()）


- 做完这一整套，就叫 1 个 training loop = 1 step
- 1 training loop = 1 step
- 1 step = 处理 1 个 batch + 更新一次参数
- 很多个 step 合起来 = 1 个 epoch（跑完一遍整个数据集）
- PyTorch 默认梯度累加，因此每个 step 前要清梯度。

In [13]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()#把模型参数的梯度清空为0，否则梯度不断累加就炸了（在一轮开头或者结束清空即可，每次backward（）前梯度干净就是等价的）

        if batch % 100 == 0:#每100个batch执行一次打印避免太多刷屏
            loss, current = loss.item(), (batch + 1) * len(X)#.item()操作吧张量转成普通数字便于打印
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")#> 表示 右对齐

We also check the model\'s performance against the test dataset to
ensure it is learning.


In [14]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)#总共有多少个batch
    model.eval()#推理（测试/验证）时必开，和no-grad搭配使用
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()#取预测概率最大的类别，与真实标签对比并转化为浮点统计正确个数
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

The training process is conducted over several iterations (*epochs*).
During each epoch, the model learns parameters to make better
predictions. We print the model\'s accuracy and loss at each epoch;
we\'d like to see the accuracy increase and the loss decrease with every
epoch.


In [15]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.306596  [   64/60000]
loss: 2.292021  [ 6464/60000]
loss: 2.271705  [12864/60000]
loss: 2.260714  [19264/60000]
loss: 2.254988  [25664/60000]
loss: 2.213193  [32064/60000]
loss: 2.223552  [38464/60000]
loss: 2.189448  [44864/60000]
loss: 2.178882  [51264/60000]
loss: 2.146859  [57664/60000]
Test Error: 
 Accuracy: 46.1%, Avg loss: 2.149761 

Epoch 2
-------------------------------
loss: 2.160726  [   64/60000]
loss: 2.154910  [ 6464/60000]
loss: 2.093401  [12864/60000]
loss: 2.110750  [19264/60000]
loss: 2.073933  [25664/60000]
loss: 1.996800  [32064/60000]
loss: 2.036241  [38464/60000]
loss: 1.951795  [44864/60000]
loss: 1.947801  [51264/60000]
loss: 1.895615  [57664/60000]
Test Error: 
 Accuracy: 56.2%, Avg loss: 1.893040 

Epoch 3
-------------------------------
loss: 1.918083  [   64/60000]
loss: 1.902143  [ 6464/60000]
loss: 1.778251  [12864/60000]
loss: 1.828575  [19264/60000]
loss: 1.734501  [25664/60000]
loss: 1.662304  [32064/600

Read more about [Training your model](optimization_tutorial.html).


------------------------------------------------------------------------


Saving Models
=============

A common way to save a model is to serialize the internal state
dictionary (containing the model parameters).


//自增笔记📒：
把模型里的参数（权重、偏置等）打包成「状态字典（state_dict）」，再序列化存成文件。

- 序列化：把内存里的模型参数，转成能永久保存的文件格式（比如 .pth / .pt），断电、重启后还能读取。
- 内部状态字典（state_dict）：模型的核心参数仓库，只存可训练的权重、偏置等关键数据，不存模型结构代码，体积小、加载快。
- 为什么这么做：这是 PyTorch 等框架的标准最佳实践，既能保存训练好的模型用于后续推理，也能断点续训。

In [16]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


Loading Models
==============

The process for loading a model includes re-creating the model structure
and loading the state dictionary into it.


In [18]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))#只加载张量权重不执行任何代码，安全写法

<All keys matched successfully>

This model can now be used to make predictions.


In [19]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


Read more about [Saving & Loading your
model](saveloadrun_tutorial.html).
